# Teste de Extração: COMEX STAT
Baixa dados de exportação/importação via CSV direto do portal da Balança Comercial.

## URLs disponíveis

**Diretório base (navegável):** https://balanca.economia.gov.br/balanca/bd/comexstat-bd/ncm/

**Exports (EXP):**
- `EXP_2023.csv`, `EXP_2024.csv`, `EXP_2025.csv`

**Imports (IMP):**
- `IMP_2023.csv`, `IMP_2024.csv`, `IMP_2025.csv`

## Problemas comuns e soluções

| Problema | Causa | Solução |
|----------|-------|----------|
| `UnicodeDecodeError` | Encoding errado | Usar `iso-8859-1` (latin1) |
| Timeout/erro de conexão | Arquivo gigante (>1GB) | Baixar arquivo primeiro, depois ler |
| Erro 503 | User-Agent genérico | Usar `Mozilla/5.0` |
| Erro SSL | Certificado não verificado | `ssl._create_unverified_context` |

In [ ]:
import pandas as pd
import urllib.request
import ssl
from pathlib import Path
import os
import time


# Ignorar erro de certificado SSL (necessário em alguns ambientes corporativos)
ssl._create_default_https_context = ssl._create_unverified_context


def baixar_comexstat(ano: int, tipo: str = "EXP"):
    """
    Baixa dados do COMEX STAT (exportação ou importação) e salva em Parquet.

    Args:
        ano: Ano dos dados (ex: 2023, 2024, 2025)
        tipo: 'EXP' para exportação, 'IMP' para importação
    """
    base_url = (
        f"https://balanca.economia.gov.br/balanca/bd/comexstat-bd/ncm/{tipo}_{ano}.csv"
    )
    output_dir = Path("data/bronze/comex_stat")
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_temp = output_dir / f"{tipo}_{ano}.csv"
    parquet_file = output_dir / f"{tipo}_{ano}.parquet"

    print(f"Baixando {tipo}_{ano}.csv ({base_url})...")

    try:
        # 1. Baixa o arquivo inteiro primeiro (mais estável para arquivos grandes)
        req = urllib.request.Request(base_url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as response:
            with open(csv_temp, "wb") as f:
                f.write(response.read())

        print(
            f"Download concluído ({csv_temp.stat().st_size / (1024*1024):.1f} MB). Lendo CSV..."
        )

        # 2. Lê com encoding correto (iso-8859-1 para caracteres especiais: ç, ã, é)
        df = pd.read_csv(
            csv_temp,
            sep=";",
            encoding="iso-8859-1",
            low_memory=False,
            dtype={"CO_ANO": "int16", "CO_MES": "int8"},  # economiza memória
        )

        # 3. Salva em Parquet (muito mais rápido e compacto)
        df.to_parquet(parquet_file, index=False, compression="snappy")

        print(f"✅ Sucesso! {len(df):,} linhas salvas em {parquet_file}")

        # Opcional: apaga o CSV bruto (economiza espaço)
        os.remove(csv_temp)

    except Exception as e:
        print(f"❌ Erro: {e}")
        if csv_temp.exists():
            print(f"Arquivo parcial em: {csv_temp}")


# === USO ===
if __name__ == "__main__":
    baixar_comexstat(2023, "IMP")   # exportações 2023
    time.sleep(4)
    # Exemplos de outras URLs derivadas:
    baixar_comexstat(2024, "IMP")
    time.sleep(4)
    baixar_comexstat(2025, "IMP")
    # baixar_comexstat(2023, "IMP")   # importações